In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import scraping_functions as sf
import importlib
import time
import re

importlib.reload(sf);

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

In [4]:
def get_page(url, headers, retries=5):
    for attempt in range(retries):
        response = requests.get(url, headers=headers)

        if response.status_code == 200:
            return response

        print(f"Attempt {attempt+1}: {response.status_code}")
        time.sleep(3)

    return response

In [4]:
url = f'https://www.transfermarkt.com/premier-league/spieltag/wettbewerb/GB1/saison_id/2025/spieltag/1'
response = get_page(url, headers)
soup = BeautifulSoup(response.content, "lxml")

## Matches csv

In [5]:
all_matches = soup.find_all('table',{'style':'border-top: 0 !important;'})

home_team = 'hauptlink zentriert no-border-links no-border-rechts hide-for-small spieltagsansicht-wappen'
away_team = 'hauptlink zentriert no-border-rechts no-border-links hide-for-small spieltagsansicht-wappen'

# SCRAPING AWAY INFORMATION
print('-----------------------------------------')
print('SCRAPING AWAY INFORMATION')
print('-----------------------------------------')

away_team_info = all_matches[5].find_all('td',{'class':away_team})

away_team_url = away_team_info[0].find('a').get('href')
away_team_id = away_team_url.split('/')[-3]
away_team_name = away_team_info[0].find('a').get('title')

display(away_team_info,away_team_url,away_team_id,away_team_name)

# SCRAPING HOME INFORMATION
print('-----------------------------------------')
print('SCRAPING HOME INFORMATION')
print('-----------------------------------------')

home_team_info = all_matches[5].find_all('td',{'class':home_team})

home_team_url = home_team_info[0].find('a').get('href')
home_team_id = home_team_url.split('/')[-3]
home_team_name = home_team_info[0].find('a').get('title')

display(home_team_info,home_team_url,home_team_id,home_team_name)

-----------------------------------------
SCRAPING AWAY INFORMATION
-----------------------------------------


[<td class="hauptlink zentriert no-border-rechts no-border-links hide-for-small spieltagsansicht-wappen">
 <a href="/manchester-city/spielplan/verein/281/saison_id/2025" title="Manchester City"><img alt="Manchester City" class="" src="https://img.a.transfermarkt.technology/wappen/small/281.png?lm=4711" title="Manchester City"/></a> </td>]

'/manchester-city/spielplan/verein/281/saison_id/2025'

'281'

'Manchester City'

-----------------------------------------
SCRAPING HOME INFORMATION
-----------------------------------------


[<td class="hauptlink zentriert no-border-links no-border-rechts hide-for-small spieltagsansicht-wappen">
 <a href="/wolverhampton-wanderers/spielplan/verein/543/saison_id/2025" title="Wolverhampton Wanderers"><img alt="Wolverhampton Wanderers" class="" src="https://img.a.transfermarkt.technology/wappen/small/543.png?lm=4711" title="Wolverhampton Wanderers"/></a> </td>]

'/wolverhampton-wanderers/spielplan/verein/543/saison_id/2025'

'543'

'Wolverhampton Wanderers'

In [6]:
match_url = all_matches[0].find('td',{'class':'spieltagsansicht-ergebnis'}).find('a').get('href')

match_id = match_url.split('/')[-1]

match_result = all_matches[0].find('span',{'class':'matchresult finished'}).string

match_info = all_matches[0].find_all('td',{'class':'zentriert no-border'})

match_day = match_info[0].find('a').get('href').split('/')[-1]
match_referee = match_info[1].find('a').string
match_attendance = match_info[2].get_text().strip()
match_attendance = re.sub('[.]','', match_attendance).split(' ')[0]
time_info = match_info[0].find('a').next_sibling.strip().removeprefix('-').strip().split(' ')
match_time = time_info[0]
match_time_period = time_info[-1]

display(match_info,match_day,match_referee,match_attendance,match_time,match_time_period,match_result,match_url,match_id)


[<td class="zentriert no-border" colspan="5">
 <div class="di"><span class="hide-for-small">Friday,</span><span class="show-for-small">Fri</span></div> <a href="/aktuell/waspassiertheute/aktuell/new/datum/2025-08-15">
                                                                 15/08/2025                                        </a>
                                                              - 9:00 PM
                             </td>,
 <td class="zentriert no-border" colspan="5">
 <span>Referee: <a href="/anthony-taylor/profil/schiedsrichter/847" title="Anthony Taylor">Anthony Taylor</a></span> </td>,
 <td class="zentriert no-border" colspan="5">
 <span class="icons_sprite icon-zuschauer-zahl" title="Attendance"> </span>
                                         60.315                                </td>]

'2025-08-15'

'Anthony Taylor'

'60315'

'9:00'

'PM'

'4:2'

'/spielbericht/index/spielbericht/4625774'

'4625774'

In [7]:
display(home_team_url,home_team_id,home_team_name,match_result,away_team_url,away_team_id,away_team_name,match_day,match_referee,match_attendance,match_time,match_time_period)

'/wolverhampton-wanderers/spielplan/verein/543/saison_id/2025'

'543'

'Wolverhampton Wanderers'

'4:2'

'/manchester-city/spielplan/verein/281/saison_id/2025'

...

'2025-08-15'

'Anthony Taylor'

'60315'

'9:00'

'PM'

## Events csv

In [8]:
match_event = all_matches[0].find_all('tr',{'class':'no-border spieltagsansicht-aktionen'})

display(match_event)

[<tr class="no-border spieltagsansicht-aktionen">
 <td class="rechts no-border-rechts spieltagsansicht"><div class="di"><div class="di nowrap"><span class="hide-for-small"><a href="/hugo-ekitike/profil/spieler/709726" title="Hugo Ekitiké">Hugo Ekitiké</a></span></div><div class="di nowrap"><span class="show-for-small"><a href="/hugo-ekitike/profil/spieler/709726" title="Hugo Ekitiké">H. Ekitiké</a></span></div></div><span class="icons_sprite icon-tor-formation" title="Minute 37: Goal"> </span></td>
 <td class="zentriert no-border-links">37'</td>
 <td class="zentriert hauptlink">1:0</td>
 <td class="zentriert no-border-rechts"> </td>
 <td class="links no-border-links"> </td>
 </tr>,
 <tr class="no-border spieltagsansicht-aktionen">
 <td class="rechts no-border-rechts spieltagsansicht"><div class="di"><div class="di nowrap"><span class="hide-for-small"><a href="/cody-gakpo/profil/spieler/434675" title="Cody Gakpo">Cody Gakpo</a></span></div><div class="di nowrap"><span class="show-for-sm

In [9]:
player_url = match_event[0].find('td',{'class':'spieltagsansicht'}).find('a').get('href')
player_id = player_url.split('/')[-1]
player_name = match_event[0].find('td',{'class':'spieltagsansicht'}).find('a').get('title')

event_type = match_event[0].find('span',{'class':'icons_sprite'}).get('class')[-1]

check = match_event[0].find('td',{'class':'zentriert hauptlink'})
event_score = None if check == None else check.string

home='links'
away='rechts'

check = lambda x: match_event[0].find('td',{'class':f'zentriert no-border-{x}'}).string
event_time_label = check(away) if check(home) == '\xa0' else check(home)

time_list = re.sub("[']",'', event_time_label).split('+')
event_time_minute = int(time_list[0])
event_time_extra = int(time_list[-1]) if len(time_list) > 1 else 0

display(player_url,player_id,player_name,event_type,event_score,event_time_label,event_time_minute,event_time_extra)

'/hugo-ekitike/profil/spieler/709726'

'709726'

'Hugo Ekitiké'

'icon-tor-formation'

'1:0'

"37'"

37

0

In [33]:
all_leagues = {
	'premier-league' : 'GB1',
	'bundesliga' : 'L1',
	'serie-a' : 'IT1',
	'laliga' : 'ES1',
	'ligue-1' : 'FR1',
	'campeonato-brasileiro-serie-a' : 'BRA1'
}

In [11]:
def get_events(headers, league, n_season, n_round):
    # Creating soup object
    url = f'https://www.transfermarkt.com/{league}/spieltag/wettbewerb/{all_leagues[league]}/saison_id/{n_season}/spieltag/{n_round}'
    response = get_page(url, headers)
    soup = BeautifulSoup(response.content, "lxml")

    # Gathering all 10 matches from the round
    all_matches = soup.find_all('table',{'style':'border-top: 0 !important;'})
    season_id = f'{all_leagues[league]}-{n_season}'

    output_list = []
    # separating matches
    for m, match in enumerate(all_matches):
        match_id = f'M-{n_season}-{n_round:02d}-{m+1:02d}'
        match_url = match.find('td',{'class':'spieltagsansicht-ergebnis'}).find('a').get('href')

        # separating events
        match_event = match.find_all('tr',{'class':'no-border spieltagsansicht-aktionen'})
        for event in match_event:
            temp = []

            # Player Information
            player_url = event.find('td',{'class':'spieltagsansicht'}).find('a').get('href')
            player_id = int(player_url.split('/')[-1])
            player_name = event.find('td',{'class':'spieltagsansicht'}).find('a').get('title')

            #Event Information
            event_type = event.find('span',{'class':'icons_sprite'}).get('class')[-1]
            check = event.find('td',{'class':'zentriert hauptlink'})
            event_score = None if check == None else check.string

            # Time Information
            home='links'
            away='rechts'

            check = lambda x: event.find('td',{'class':f'zentriert no-border-{x}'}).string
            event_time_label = check(away) if check(home) == '\xa0' else check(home)

            time_list = re.sub("[']",'', event_time_label).split('+')
            event_time_minute = int(time_list[0])
            event_time_extra = int(time_list[-1]) if len(time_list) > 1 else 0

            # Appending to the output
            temp.append(season_id)
            temp.append(match_id)

            temp.append(match_url)
            temp.append(player_url)
            temp.append(player_id)
            temp.append(player_name)
            temp.append(event_type)
            temp.append(event_score)
            temp.append(event_time_label)
            temp.append(event_time_minute)
            temp.append(event_time_extra)

            output_list.append(temp)
    output_list.insert(0,['season_id','match_id','match_url','player_url','player_id','player_name','event_type','event_score','event_time_label','event_time_minute','event_time_extra'])
    return output_list

In [12]:
ltest = get_events(headers,'premier-league',2025,1)

df = pd.DataFrame(ltest[1:],columns=ltest[0])

display(df)

,season_id,match_id,match_url,player_url,player_id,player_name,event_type,event_score,event_time_label,event_time_minute,event_time_extra
0,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/hugo-ekitike/profil/spieler/709726,709726,Hugo Ekitiké,icon-tor-formation,1:0,37',37,0
1,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/cody-gakpo/profil/spieler/434675,434675,Cody Gakpo,icon-tor-formation,2:0,49',49,0
2,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/antoine-semenyo/profil/spieler/583255,583255,Antoine Semenyo,icon-tor-formation,2:1,64',64,0
3,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/antoine-semenyo/profil/spieler/583255,583255,Antoine Semenyo,icon-tor-formation,2:2,76',76,0
4,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/federico-chiesa/profil/spieler/341092,341092,Federico Chiesa,icon-tor-formation,3:2,88',88,0
5,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/mohamed-salah/profil/spieler/148455,148455,Mohamed Salah,icon-tor-formation,4:2,90+4',90,4
6,GB1-2025,M-2025-01-02,/spielbericht/index/spielbericht/4625775,/ezri-konsa/profil/spieler/413403,413403,Ezri Konsa,icon-rotekarte-formation,None,66',66,0
7,GB1-2025,M-2025-01-03,/spielbericht/index/spielbericht/4625777,/matt-oriley/profil/spieler/406634,406634,Matt O'Riley,icon-elfmeter-formation,1:0,55',55,0
8,GB1-2025,M-2025-01-03,/spielbericht/index/spielbericht/4625777,/rodrigo-muniz/profil/spieler/735571,735571,Rodrigo Muniz,icon-tor-formation,1:1,90+6',90,6
9,GB1-2025,M-2025-01-04,/spielbericht/index/spielbericht/4625778,/eliezer-mayenda/profil/spieler/967346,967346,Eliezer Mayenda,icon-tor-formation,1:0,61',61,0


In [33]:
def get_matches(headers, league, n_season, n_round):
    # Creating soup object
    url = f'https://www.transfermarkt.com/{league}/spieltag/wettbewerb/{all_leagues[league]}/saison_id/{n_season}/spieltag/{n_round}'
    response = get_page(url, headers)
    soup = BeautifulSoup(response.content, "lxml")

    # Gathering all 10 matches from the round
    all_matches = soup.find_all('table',{'style':'border-top: 0 !important;'})

    # Different calls for home and away teams
    home_team = 'hauptlink zentriert no-border-links no-border-rechts hide-for-small spieltagsansicht-wappen'
    away_team = 'hauptlink zentriert no-border-rechts no-border-links hide-for-small spieltagsansicht-wappen'

    season_id = f'{all_leagues[league]}-{n_season}'

    output_list = []
    # separating matches
    for m, match in enumerate(all_matches):
        match_id = f'M-{n_season}-{n_round:02d}-{m+1:02d}'

        match_url = match.find('td',{'class':'spieltagsansicht-ergebnis'}).find('a').get('href')

        temp = []
        # Information from each match
        # Scraping Away Information
        away_team_info = match.find_all('td',{'class':away_team})

        away_team_url = away_team_info[0].find('a').get('href')
        away_team_id = int(away_team_url.split('/')[-3])
        away_team_name = away_team_info[0].find('a').get('title')

        # Scraping Home Information
        home_team_info = match.find_all('td',{'class':home_team})

        home_team_url = home_team_info[0].find('a').get('href')
        home_team_id = int(home_team_url.split('/')[-3])
        home_team_name = home_team_info[0].find('a').get('title')

        # Scraping Result
        match_result = match.find('span',{'class':'matchresult finished'}).string

        # Scraping Addicional Information
        match_info = match.find_all('td',{'class':'zentriert no-border'})

        match_day = match_info[0].find('a').get('href').split('/')[-1]
        match_referee = match_info[1].find('a').string
        match_attendance = match_info[2].get_text().strip()
        match_attendance = int(re.sub('[.]','', match_attendance).split(' ')[0])
        time_info = match_info[0].find('a').next_sibling.strip().removeprefix('-').strip().split(' ')
        match_time = time_info[0]
        match_time_period = time_info[-1]

        # Appending to the output
        temp.append(season_id)
        temp.append(match_id)

        temp.append(match_url)
        temp.append(home_team_url)
        temp.append(home_team_id)
        temp.append(home_team_name)
        temp.append(match_result)
        temp.append(away_team_url)
        temp.append(away_team_id)
        temp.append(away_team_name)
        temp.append(match_day)
        temp.append(match_referee)
        temp.append(match_attendance)
        temp.append(match_time)
        temp.append(match_time_period)

        output_list.append(temp)
    output_list.insert(0,['season_id','match_id','match_url','home_team_url','home_team_id','home_team_name','match_result','away_team_url','away_team_id','away_team_name','match_day','match_referee','match_attendance','match_time','match_time_period'])
    return output_list

In [34]:
ltest2 = get_matches(headers,'premier-league',2025,1)

df2 = pd.DataFrame(ltest2[1:],columns=ltest2[0])

display(df2)

,season_id,match_id,match_url,home_team_url,home_team_id,home_team_name,match_result,away_team_url,away_team_id,away_team_name,match_day,match_referee,match_attendance,match_time,match_time_period
0,GB1-2025,M-2025-01-01,/spielbericht/index/spielbericht/4625774,/fc-liverpool/spielplan/verein/31/saison_id/2025,31,Liverpool FC,4:2,/afc-bournemouth/spielplan/verein/989/saison_i...,989,AFC Bournemouth,2025-08-15,Anthony Taylor,60315,9:00,PM
1,GB1-2025,M-2025-01-02,/spielbericht/index/spielbericht/4625775,/aston-villa/spielplan/verein/405/saison_id/2025,405,Aston Villa,0:0,/newcastle-united/spielplan/verein/762/saison_...,762,Newcastle United,2025-08-16,Craig Pawson,42526,1:30,PM
2,GB1-2025,M-2025-01-03,/spielbericht/index/spielbericht/4625777,/brighton-amp-hove-albion/spielplan/verein/123...,1237,Brighton & Hove Albion,1:1,/fc-fulham/spielplan/verein/931/saison_id/2025,931,Fulham FC,2025-08-16,Samuel Barrott,31478,4:00,PM
3,GB1-2025,M-2025-01-04,/spielbericht/index/spielbericht/4625778,/afc-sunderland/spielplan/verein/289/saison_id...,289,Sunderland AFC,3:0,/west-ham-united/spielplan/verein/379/saison_i...,379,West Ham United,2025-08-16,Robert Jones,46233,4:00,PM
4,GB1-2025,M-2025-01-05,/spielbericht/index/spielbericht/4625779,/tottenham-hotspur/spielplan/verein/148/saison...,148,Tottenham Hotspur,3:0,/fc-burnley/spielplan/verein/1132/saison_id/2025,1132,Burnley FC,2025-08-16,Michael Oliver,61077,4:00,PM
5,GB1-2025,M-2025-01-06,/spielbericht/index/spielbericht/4625780,/wolverhampton-wanderers/spielplan/verein/543/...,543,Wolverhampton Wanderers,0:4,/manchester-city/spielplan/verein/281/saison_i...,281,Manchester City,2025-08-16,Jarred Gillett,31118,6:30,PM
6,GB1-2025,M-2025-01-07,/spielbericht/index/spielbericht/4625776,/nottingham-forest/spielplan/verein/703/saison...,703,Nottingham Forest,3:1,/fc-brentford/spielplan/verein/1148/saison_id/...,1148,Brentford FC,2025-08-17,Peter Bankes,29949,3:00,PM
7,GB1-2025,M-2025-01-08,/spielbericht/index/spielbericht/4625781,/fc-chelsea/spielplan/verein/631/saison_id/2025,631,Chelsea FC,0:0,/crystal-palace/spielplan/verein/873/saison_id...,873,Crystal Palace,2025-08-17,Darren England,39678,3:00,PM
8,GB1-2025,M-2025-01-09,/spielbericht/index/spielbericht/4625782,/manchester-united/spielplan/verein/985/saison...,985,Manchester United,0:1,/fc-arsenal/spielplan/verein/11/saison_id/2025,11,Arsenal FC,2025-08-17,Simon Hooper,73475,5:30,PM
9,GB1-2025,M-2025-01-10,/spielbericht/index/spielbericht/4625783,/leeds-united/spielplan/verein/399/saison_id/2025,399,Leeds United,1:0,/fc-everton/spielplan/verein/29/saison_id/2025,29,Everton FC,2025-08-18,Chris Kavanagh,36820,9:00,PM


In [15]:
url2 = f'https://www.transfermarkt.com/premier-league/torschuetzenliste/wettbewerb/GB1/saison_id/2025/altersklasse/alle/detailpos//page/1'
response = get_page(url2, headers)
soup = BeautifulSoup(response.content, "lxml")

In [ ]:
all_content = soup.find_all('tbody')

content = all_content[1].find_all('tr',{'class':['odd','even']})

i = 5

td_player_info = content[i].find_all('td',{'class':'zentriert'})

leaderboard_pos = td_player_info[0].string
country_name = td_player_info[1].find('img').get('title')
player_age = td_player_info[2].string

if td_player_info[3].string == None:
    team_name = td_player_info[3].find('a').get('title')
    team_url = td_player_info[3].find('a').get('href')
    team_id = team_url.split('/')[-3]
else:
    team_name = td_player_info[3].string
    team_url = None 
    team_id = 0

player_name = td_player_info[4].find('a').get('title')
player_url = td_player_info[4].find('a').get('href')
player_id = player_url.split('/')[-5]
macthes_played = td_player_info[4].string
goals = td_player_info[5].string





display(td_player_info,leaderboard_pos,country_name,player_age,team_name,team_url,team_id,player_name,player_url,player_id,macthes_played,goals)

[<td class="zentriert">6</td>,
 <td class="zentriert"><img alt="England" class="flaggenrahmen" src="https://img.a.transfermarkt.technology/flagge/verysmall/189.png?lm=4711" title="England"/><br/><img alt="Jamaica" class="flaggenrahmen" src="https://img.a.transfermarkt.technology/flagge/verysmall/76.png?lm=4711" title="Jamaica"/></td>,
 <td class="zentriert">26</td>,
 <td class="zentriert"><a href="/nottingham-forest/startseite/verein/703/saison_id/2025" title="Nottingham Forest"><img alt="Nottingham Forest" class="" src="https://img.a.transfermarkt.technology/wappen/verysmall/703.png?lm=4711" title="Nottingham Forest"/></a></td>,
 <td class="zentriert"><a href="/morgan-gibbs-white/leistungsdaten/spieler/429014/saison/2025/wettbewerb/GB1" title="Morgan Gibbs-White">37</a></td>,
 <td class="zentriert hauptlink"><a href="/morgan-gibbs-white/leistungsdaten/spieler/429014/saison/2025/wettbewerb/GB1" title="Morgan Gibbs-White">15</a></td>]

'6'

'England'

'26'

'Nottingham Forest'

...

'Morgan Gibbs-White'

'/morgan-gibbs-white/leistungsdaten/spieler/429014/saison/2025/wettbewerb/GB1'

'429014'

'37'

'15'

In [18]:
pages_info = soup.find_all('div', {'class':'pager'})
last_page_link = pages_info[0].find_all('li',{'class':'tm-pagination__list-item tm-pagination__list-item--icon-last-page'})
last_page_number = last_page_link[0].find('a').get('href').split('/')[-1]

display(last_page_number)

'12'

In [28]:
def get_top_scorers(headers, league, n_season):
    # Creating soup object
    url = f'https://www.transfermarkt.com/{league}/torschuetzenliste/wettbewerb/{all_leagues[league]}/saison_id/{n_season}/altersklasse/alle/detailpos//page/1'
    response = get_page(url, headers)
    soup = BeautifulSoup(response.content, "lxml")

    # Finding last page
    pages_info = soup.find_all('div', {'class':'pager'})
    last_page_link = pages_info[0].find_all('li',{'class':'tm-pagination__list-item tm-pagination__list-item--icon-last-page'})
    last_page_number = last_page_link[0].find('a').get('href').split('/')[-1]

    output_list = []
    # Scraping all pages
    for n_page in range(1,int(last_page_number)+1):
        url = f'https://www.transfermarkt.com/{league}/torschuetzenliste/wettbewerb/{all_leagues[league]}/saison_id/{n_season}/altersklasse/alle/detailpos//page/{n_page}'
        response = get_page(url, headers)
        soup = BeautifulSoup(response.content, "lxml")

        all_content = soup.find_all('tbody')
        content = all_content[1].find_all('tr',{'class':['odd','even']})

        season_id = f'{all_leagues[league]}-{n_season}'

        for row in content:
            temp = []

            td_player_info = row.find_all('td',{'class':'zentriert'})

            leaderboard_pos = int(td_player_info[0].string)
            country_name = td_player_info[1].find('img').get('title')
            player_age = int(td_player_info[2].string)

            if td_player_info[3].string == None:
                team_name = td_player_info[3].find('a').get('title')
                team_url = td_player_info[3].find('a').get('href')
                team_id = int(team_url.split('/')[-3])
            else:
                team_name = td_player_info[3].string
                team_url = None 
                team_id = 0

            player_name = td_player_info[4].find('a').get('title')
            player_url = td_player_info[4].find('a').get('href')
            player_id = int(player_url.split('/')[-5])
            macthes_played = int(td_player_info[4].string)
            goals = int(td_player_info[5].string)

            
            temp.append(season_id)

            temp.append(player_url)
            temp.append(player_id)
            temp.append(player_name)
            temp.append(player_age)
            temp.append(country_name)
            temp.append(team_url)
            temp.append(team_id)
            temp.append(team_name)
            temp.append(leaderboard_pos)
            temp.append(macthes_played)
            temp.append(goals)
            
            output_list.append(temp)
    output_list.insert(0,['season_id','player_url','player_id','player_name','player_age','country_name','team_url','team_id','team_name','leaderboard_pos','macthes_played','goals'])
    return output_list


In [29]:
ltest3 = get_top_scorers(headers,'premier-league',2024)

df3 = pd.DataFrame(ltest3[1:],columns=ltest3[0])

display(df3)

,season_id,player_url,player_id,player_name,player_age,country_name,team_url,team_id,team_name,leaderboard_pos,macthes_played,goals
0,GB1-2024,/mohamed-salah/leistungsdaten/spieler/148455/s...,148455,Mohamed Salah,32,Egypt,/fc-liverpool/startseite/verein/31/saison_id/2024,31,Liverpool FC,1,38,29
1,GB1-2024,/alexander-isak/leistungsdaten/spieler/349066/...,349066,Alexander Isak,25,Sweden,/newcastle-united/startseite/verein/762/saison...,762,Newcastle United,2,34,23
2,GB1-2024,/erling-haaland/leistungsdaten/spieler/418560/...,418560,Erling Haaland,24,Norway,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,3,31,22
3,GB1-2024,/bryan-mbeumo/leistungsdaten/spieler/413039/sa...,413039,Bryan Mbeumo,25,Cameroon,/fc-brentford/startseite/verein/1148/saison_id...,1148,Brentford FC,4,38,20
4,GB1-2024,/chris-wood/leistungsdaten/spieler/108725/sais...,108725,Chris Wood,33,New Zealand,/nottingham-forest/startseite/verein/703/saiso...,703,Nottingham Forest,5,36,20
...,...,...,...,...,...,...,...,...,...,...,...,...
266,GB1-2024,/ross-stewart/leistungsdaten/spieler/447995/sa...,447995,Ross Stewart,28,Scotland,/fc-southampton/startseite/verein/180/saison_i...,180,Southampton FC,267,12,1
267,GB1-2024,/nico-gonzalez/leistungsdaten/spieler/466805/s...,466805,Nico González,23,Spain,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,268,11,1
268,GB1-2024,/ben-chilwell/leistungsdaten/spieler/316125/sa...,316125,Ben Chilwell,28,England,None,0,for 2 clubs,269,8,1
269,GB1-2024,/ferdi-kadioglu/leistungsdaten/spieler/369316/...,369316,Ferdi Kadıoğlu,25,Türkiye,/brighton-amp-hove-albion/startseite/verein/12...,1237,Brighton & Hove Albion,270,6,1


### SQUAD

In [58]:
#f'https://www.transfermarkt.com/{league}/startseite/wettbewerb/{all_leagues[league]}/plus/?saison_id={n_season}'
url3 = 'https://www.transfermarkt.com/premier-league/startseite/wettbewerb/GB1/plus/?saison_id=2023'
response = get_page(url3, headers)
soup = BeautifulSoup(response.content, "lxml")

In [59]:
all_info = soup.find_all('table',{'class':'items'})
table_info = all_info[0].find_all('tr',{'class':['odd','even']})
display(table_info)

[<tr class="odd">
 <td class="zentriert no-border-rechts"><a href="/manchester-city/startseite/verein/281/saison_id/2023" title="Manchester City"><img alt="Manchester City" class="tiny_wappen" src="https://img.a.transfermarkt.technology/wappen/tiny/281.png?lm=4711" title="Manchester City"/></a></td><td class="hauptlink no-border-links"><a href="/manchester-city/startseite/verein/281/saison_id/2023" title="Manchester City">Manchester City</a> <a href="#"><img alt="English Champion 22/23" class="tabelle-erfolg" src="https://img.a.transfermarkt.technology/erfolge/mini/12.png?lm=4711" title="English Champion 22/23"/></a><a href="#"><img alt="FA Cup Winner 22/23" class="tabelle-erfolg" src="https://img.a.transfermarkt.technology/erfolge/mini/29.png?lm=4711" title="FA Cup Winner 22/23"/></a><a href="#"><img alt="UEFA Champions League winner 22/23" class="tabelle-erfolg" src="https://img.a.transfermarkt.technology/erfolge/mini/4.png?lm=4711" title="UEFA Champions League winner 22/23"/></a></t

In [67]:
team_info = table_info[0].find_all('a')

team_url = team_info[0].get('href')
team_id = int(team_url.split('/')[-3])
team_name = team_info[1].string
team_squad = int(team_info[-2].string)
team_value = team_info[-1].string

abv_index = team_value[-1]
if abv_index == 'n': team_value_int = int(float(team_value.replace('€', '').replace('bn', '')) * 1_000_000_000)
elif abv_index == 'm': team_value_int = int(float(team_value.replace('€', '').replace('m', '')) * 1_000_000)
elif abv_index == 'k': team_value_int = int(float(team_value.replace('€', '').replace('k', '')) * 1_000)
else: team_value_int = int(team_value.replace('€', ''))

add_info = table_info[0].find_all('td',{'class':'zentriert'})
team_avg_age = float(add_info[-2].string)
team_foreigners = int(add_info[-1].string)

# bn,m,k
#team_value_int = 0


display(team_info,add_info,team_url,team_id,team_name,team_squad,team_value,team_value_int,team_avg_age,team_foreigners)

[<a href="/manchester-city/startseite/verein/281/saison_id/2023" title="Manchester City"><img alt="Manchester City" class="tiny_wappen" src="https://img.a.transfermarkt.technology/wappen/tiny/281.png?lm=4711" title="Manchester City"/></a>,
 <a href="/manchester-city/startseite/verein/281/saison_id/2023" title="Manchester City">Manchester City</a>,
 <a href="#"><img alt="English Champion 22/23" class="tabelle-erfolg" src="https://img.a.transfermarkt.technology/erfolge/mini/12.png?lm=4711" title="English Champion 22/23"/></a>,
 <a href="#"><img alt="FA Cup Winner 22/23" class="tabelle-erfolg" src="https://img.a.transfermarkt.technology/erfolge/mini/29.png?lm=4711" title="FA Cup Winner 22/23"/></a>,
 <a href="#"><img alt="UEFA Champions League winner 22/23" class="tabelle-erfolg" src="https://img.a.transfermarkt.technology/erfolge/mini/4.png?lm=4711" title="UEFA Champions League winner 22/23"/></a>,
 <a href="/manchester-city/kader/verein/281/saison_id/2023" title="Manchester City">36</a>

[<td class="zentriert no-border-rechts"><a href="/manchester-city/startseite/verein/281/saison_id/2023" title="Manchester City"><img alt="Manchester City" class="tiny_wappen" src="https://img.a.transfermarkt.technology/wappen/tiny/281.png?lm=4711" title="Manchester City"/></a></td>,
 <td class="zentriert"><a href="/manchester-city/kader/verein/281/saison_id/2023" title="Manchester City">36</a></td>,
 <td class="zentriert">25.7</td>,
 <td class="zentriert">21</td>]

'/manchester-city/startseite/verein/281/saison_id/2023'

281

'Manchester City'

36

'€1.46bn'

1460000000

25.7

21

In [ ]:
def get_squad(headers, league, n_season):
    url = f'https://www.transfermarkt.com/{league}/startseite/wettbewerb/{all_leagues[league]}/plus/?saison_id={n_season}'
    response = get_page(url, headers)
    soup = BeautifulSoup(response.content, "lxml")

    season_id = f'{all_leagues[league]}-{n_season}'

    # separating the tables
    all_info = soup.find_all('table',{'class':'items'})

    output_list = []
    # all teams
    table_info = all_info[0].find_all('tr',{'class':['odd','even']})
    for row in table_info:
        team_info = row.find_all('a')

        team_url = team_info[0].get('href')
        team_id = int(team_url.split('/')[-3])
        team_name = team_info[1].string
        team_squad = int(team_info[-2].string)
        team_value = team_info[-1].string

        abv_index = team_value[-1]
        if abv_index == 'n': team_value_int = int(float(team_value.replace('€', '').replace('bn', '')) * 1_000_000_000)
        elif abv_index == 'm': team_value_int = int(float(team_value.replace('€', '').replace('m', '')) * 1_000_000)
        elif abv_index == 'k': team_value_int = int(float(team_value.replace('€', '').replace('k', '')) * 1_000)
        else: team_value_int = int(team_value.replace('€', ''))

        add_info = row.find_all('td',{'class':'zentriert'})
        team_avg_age = float(add_info[-2].string)
        team_foreigners = int(add_info[-1].string)

        temp = {
            'season_id': season_id,
            'team_url': team_url,
            'team_id': team_id,
            'team_name': team_name,
            'team_squad': team_squad,
            'team_value': team_value,
            'team_value_int': team_value_int,
            'team_avg_age': team_avg_age,
            'team_foreigners': team_foreigners
        }
        
        output_list.append(temp)

    return output_list
    

In [78]:
ltest4 = get_squad(headers,'premier-league',2025)

display(ltest4)

[{'season_id': 'GB1-2025',
  'team_url': '/manchester-city/startseite/verein/281/saison_id/2025',
  'team_id': 281,
  'team_name': 'Manchester City',
  'team_squad': 43,
  'team_value': '€1.39bn',
  'team_value_int': 1390000000,
  'team_avg_age': 25.2,
  'team_foreigners': 24},
 {'season_id': 'GB1-2025',
  'team_url': '/fc-arsenal/startseite/verein/11/saison_id/2025',
  'team_id': 11,
  'team_name': 'Arsenal FC',
  'team_squad': 40,
  'team_value': '€1.33bn',
  'team_value_int': 1330000000,
  'team_avg_age': 23.9,
  'team_foreigners': 20},
 {'season_id': 'GB1-2025',
  'team_url': '/fc-chelsea/startseite/verein/631/saison_id/2025',
  'team_id': 631,
  'team_name': 'Chelsea FC',
  'team_squad': 42,
  'team_value': '€1.15bn',
  'team_value_int': 1150000000,
  'team_avg_age': 22.4,
  'team_foreigners': 22},
 {'season_id': 'GB1-2025',
  'team_url': '/fc-liverpool/startseite/verein/31/saison_id/2025',
  'team_id': 31,
  'team_name': 'Liverpool FC',
  'team_squad': 45,
  'team_value': '€980.5

In [77]:
premier_squad = []

# transfermarkt only contains values for team_value from 2004
for season in range(2024,2026):
    squad_data = get_squad(headers,'premier-league',season)
    premier_squad.extend(squad_data)

df_premier_squad = pd.DataFrame(premier_squad)
display(df_premier_squad)

,season_id,team_url,team_id,team_name,team_squad,team_value,team_value_int,team_avg_age,team_foreigners
0,GB1-2024,/manchester-city/startseite/verein/281/saison_...,281,Manchester City,44,€1.36bn,1360000000,25.6,27
1,GB1-2024,/fc-chelsea/startseite/verein/631/saison_id/2024,631,Chelsea FC,58,€1.19bn,1190000000,22.4,32
2,GB1-2024,/fc-arsenal/startseite/verein/11/saison_id/2024,11,Arsenal FC,42,€1.16bn,1160000000,24.5,26
3,GB1-2024,/fc-liverpool/startseite/verein/31/saison_id/2024,31,Liverpool FC,35,€950.95m,950950000,24.7,21
4,GB1-2024,/manchester-united/startseite/verein/985/saiso...,985,Manchester United,49,€825.00m,825000000,24.0,29
5,GB1-2024,/tottenham-hotspur/startseite/verein/148/saiso...,148,Tottenham Hotspur,41,€777.30m,777300000,24.0,23
6,GB1-2024,/aston-villa/startseite/verein/405/saison_id/2024,405,Aston Villa,41,€680.55m,680550000,25.2,24
7,GB1-2024,/newcastle-united/startseite/verein/762/saison...,762,Newcastle United,36,€677.53m,677530000,27.5,15
8,GB1-2024,/brighton-amp-hove-albion/startseite/verein/12...,1237,Brighton & Hove Albion,49,€667.58m,667580000,24.2,33
9,GB1-2024,/crystal-palace/startseite/verein/873/saison_i...,873,Crystal Palace,40,€526.85m,526850000,26.2,18
